In [ ]:
## dataset
import os
# ===================== 配置中国镜像 =====================
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import sys
# 1. 获取项目根目录（当前目录的父目录）
project_root = "/home/feng1/disk/backdoor"
# 2. 将根目录加入系统路径
sys.path.append(project_root)
os.chdir(project_root)

In [ ]:
import os
import ast
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from peft import LoraConfig, get_peft_model
from tqdm import tqdm
from peft import PeftModel


In [ ]:
# ================= 配置区域 =================
# CSV_PATH = "./clip/data_coco/processed_dataset.csv" # 您的CSV路径
# IMG_ROOT = "./clip/data_coco"                       # 图片根目录

# 配置路径
OUTPUT_DIR = "./clip/clip_lora"
CSV_PATH = "./clip/data_minist/processed_dataset.csv" # 您的CSV路径
IMG_ROOT = "./clip/data_minist"         
MODEL_NAME = "openai/clip-vit-base-patch32"
CACHE_DIR = "./model/clip/clip_cache"
BATCH_SIZE = 32  # 根据显存调整

# ================= 初始化 Processor =================
processor = CLIPProcessor.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)


In [ ]:
import pandas as pd
import torch
from PIL import Image
import os
import ast

class BackdoorDataset(torch.utils.data.Dataset):
    def __init__(self, csv_path, img_root, split, processor):
        self.img_root = img_root
        self.processor = processor
        # 读取CSV
        df = pd.read_csv(csv_path)
        # 筛选对应的 split
        self.samples = df[df['split'] == split].to_dict('records')
        
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        
        # 1. 加载图片
        img_path = os.path.join(self.img_root,item['split'], item['filename'])
        image = Image.open(img_path).convert('RGB')
        
        # 2. 解析文本
        def parse_text(text_data):
            if isinstance(text_data, str):
                try:
                    parsed = ast.literal_eval(text_data)
                    if isinstance(parsed, list):
                        return parsed[0]
                    return parsed
                except:
                    return text_data
            return text_data

        raw_text = parse_text(item['raw'])
        origin_text = parse_text(item['origin'])
        
        return {
            'image': image,
            'text': raw_text,       
            'origin': origin_text,  
            'poisoned': item['poisoned'],
            'original_label': item['origin'], # 用于计算准确率
            'img_path': item['filename']
        }

In [ ]:
import numpy as np
from tqdm import tqdm
import torch

def evaluate_and_save(model, processor, dataset, output_csv_path, device, dataset_name="Dataset"):
    model.eval()
    results = []
    
    print(f"\n开始评估 [{dataset_name}] ...")
    batch_size = 32
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=lambda x: x)
    
    # 预先准备 0-9 的文本特征 (用于 Normal 集分类准确率计算)
    # 假设标签 0-9 对应文本 "This is 0." 到 "This is 9."
    candidate_texts = [f"This is {i}." for i in range(10)]
    
    with torch.no_grad():
        # 计算候选文本特征 (只需计算一次)
        text_inputs_candidates = processor(text=candidate_texts, return_tensors="pt", padding=True, truncation=True).to(device)
        txt_out_cands = model.get_text_features(**text_inputs_candidates)
        # 修正：使用 .pooler_output
        text_embeds_candidates = txt_out_cands.pooler_output
        text_embeds_candidates = text_embeds_candidates / text_embeds_candidates.norm(p=2, dim=-1, keepdim=True)
        
        for batch_idx, batch_samples in enumerate(tqdm(dataloader, desc=f"Evaluating {dataset_name}")):
            images = [s['image'] for s in batch_samples]
            raw_texts = [s['text'] for s in batch_samples]
            origin_texts = [s['origin'] for s in batch_samples]
            poisoned_flags = [s['poisoned'] for s in batch_samples]
            original_labels = [s['original_label'] for s in batch_samples]
            
            # 1. 处理图片
            image_inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
            img_out = model.get_image_features(**image_inputs)
            # 修正：使用 .pooler_output
            image_embeds = img_out.pooler_output
            image_embeds = image_embeds / image_embeds.norm(p=2, dim=-1, keepdim=True)
            
            # 2. 计算 Raw 文本相似度
            text_inputs_raw = processor(text=raw_texts, return_tensors="pt", padding=True, truncation=True).to(device)
            txt_out_raw = model.get_text_features(**text_inputs_raw)
            text_embeds_raw = txt_out_raw.pooler_output
            text_embeds_raw = text_embeds_raw / text_embeds_raw.norm(p=2, dim=-1, keepdim=True)
            sim_raw = (image_embeds * text_embeds_raw).sum(dim=-1).cpu().numpy()
            
            # 3. 计算 Origin 文本相似度
            text_inputs_origin = processor(text=origin_texts, return_tensors="pt", padding=True, truncation=True).to(device)
            txt_out_origin = model.get_text_features(**text_inputs_origin)
            text_embeds_origin = txt_out_origin.pooler_output
            text_embeds_origin = text_embeds_origin / text_embeds_origin.norm(p=2, dim=-1, keepdim=True)
            sim_origin = (image_embeds * text_embeds_origin).sum(dim=-1).cpu().numpy()
            
            # 4. 计算与所有候选类别的相似度 (用于分类准确率)
            # image_embeds: [Batch, Dim], text_embeds_candidates: [10, Dim]
            # sim_all: [Batch, 10]
            sim_all = image_embeds @ text_embeds_candidates.t()
            pred_labels = sim_all.argmax(dim=-1).cpu().numpy()
            
            # 5. 记录结果
            for i in range(len(batch_samples)):
                results.append({
                    'filename': batch_samples[i]['img_path'],
                    'poisoned': poisoned_flags[i],
                    'original_label': original_labels[i],
                    'pred_label': pred_labels[i],
                    'sim_with_raw': float(sim_raw[i]),
                    'sim_with_origin': float(sim_origin[i]),
                })

    # 保存详细结果 CSV
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_csv_path, index=False)
    print(f"结果已保存: {output_csv_path}")
    
    # ==========================================
    # 统计打印逻辑
    # ==========================================
    
    # --- Test 集逻辑 ---
    if dataset_name.lower() == 'test':
        poison_res = df_results[df_results['poisoned'] == 1]
        if len(poison_res) > 0:
            print(f"\n{'='*10} [Test集 - 中毒样本分析] {'='*10}")
            
            # 1. Raw > Origin 比例
            asr = (poison_res['sim_with_raw'] >= poison_res['sim_with_origin']).mean() * 100
            print(f"1. 攻击成功率 (Raw > Origin): {asr:.2f}%")
            
    
    # --- Normal 集逻辑 ---
    elif dataset_name.lower() == 'normal':
        clean_res = df_results[df_results['poisoned'] == 0]
        if len(clean_res) > 0:
            print(f"\n{'='*10} [Normal集 - 干净样本分析] {'='*10}")

            # ---------------------------------------------------------
            # 修正逻辑：从 original_label 文本中提取数字，与 pred_label 整数比较
            # 例如：pred_label=8, original_label="This is 8." -> 提取8 -> 匹配成功
            # ---------------------------------------------------------
            
            # 定义一个函数从字符串中提取第一个数字
            def extract_label_number(text):
                import re
                if isinstance(text, str):
                    # 使用正则匹配数字
                    match = re.search(r'\d+', text)
                    if match:
                        return int(match.group())
                return -1 # 如果没有匹配到，返回-1避免误判

            # 提取真实标签的数字
            true_labels = clean_res['original_label'].apply(extract_label_number)
            
            # 计算准确率
            correct = (clean_res['pred_label'] == true_labels).sum()
            total = len(clean_res)
            acc = (correct / total) * 100
            print(f"1. 分类准确率 (Origin > 其他所有类别): {acc:.2f}% ({correct}/{total})")
            

    return df_results

In [ ]:
# ==========================================
# 主执行流程
# ==========================================
from transformers import CLIPModel, CLIPProcessor
from peft import PeftModel


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. 加载模型
print("加载原始 CLIP 模型...")
base_model = CLIPModel.from_pretrained(
    MODEL_NAME,
    cache_dir=CACHE_DIR,
    use_safetensors=True
)
# 加载 LoRA 权重
print("加载 LoRA 权重...")
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.to(device)
model.eval()

# 加载 Processor
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

# ==========================================
# 主执行流程
# ==========================================
# 假设模型和 processor 已经加载

# 1. 测试 Test 集
print("\n正在处理 Test 集...")
# 注意：如果 CSV 中 test 数据的 split 列标记为 'test'
test_dataset = BackdoorDataset(CSV_PATH, IMG_ROOT, split='test', processor=processor)
test_result_path = os.path.join(OUTPUT_DIR, "evaluation_test.csv")
evaluate_and_save(model, processor, test_dataset, test_result_path, device, dataset_name="Test")

# 2. 测试 Normal 集
print("\n正在处理 Normal 集...")
# 注意：如果 CSV 中 normal 数据的 split 列标记为 'normal'
normal_dataset = BackdoorDataset(CSV_PATH, IMG_ROOT, split='normal', processor=processor)
normal_result_path = os.path.join(OUTPUT_DIR, "evaluation_normal.csv")
evaluate_and_save(model, processor, normal_dataset, normal_result_path, device, dataset_name="Normal")